# Jaipur Flood Vulnerability Index

**Which parts of Jaipur are most likely to waterlog, and which of those can least afford it?**

This notebook builds the answer from scratch using free public data. Run the cells in
order. Nothing needs installing on your computer — Colab does the work.

### What you will actually do

1. Download 10 years of real rainfall for Jaipur and find the worst storms
2. Download terrain data and work out where water flows
3. Download Jaipur's drains, roads and buildings from OpenStreetMap
4. Combine it all into a risk score for every 300 m square of the city
5. Test whether the model survives changing your own assumptions
6. Check it against places that really flooded

### Before you start

You do not need to understand every line. You *do* need to understand what each **step**
is doing and why, because that is what someone will ask you about. Every section starts
with a plain-English explanation. Read those.

Total run time: about 15 minutes, most of it waiting for downloads.

---
## 0. Setup

This pulls the project code and installs the two libraries Colab does not already have.

`!` at the start of a line means "run this as a terminal command, not as Python".

In [ ]:
# The repo this notebook pulls its code from.
GITHUB_USER = "samyakshah113"

import os

if not os.path.exists("jaipur-flood-index"):
    !git clone https://github.com/{GITHUB_USER}/jaipur-flood-index.git

%cd jaipur-flood-index

!pip install -q folium branca

print("\nReady.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, features, index, mapping, terrain, validate

# Make the charts readable rather than tiny and grey.
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"Study area: {config.STUDY_AREA['name']}")
print(f"Covering {config.STUDY_AREA['min_lat']}-{config.STUDY_AREA['max_lat']} N, "
      f"{config.STUDY_AREA['min_lon']}-{config.STUDY_AREA['max_lon']} E")

---
## 1. How much does it actually rain in Jaipur?

Start here because it is fast and it gives you real numbers immediately.

We are pulling **measured daily rainfall** from Open-Meteo's historical archive, which is
built on ERA5 — the European weather centre's reconstruction of past weather, blending
station readings, satellite data and a physics model.

Two things to watch for in the output:

- **The monsoon share.** Jaipur's annual rainfall total sounds manageable. The fact that
  almost all of it arrives in about six weeks is the whole problem. Drainage sized for
  the average is drainage that fails every single year.
- **The ten wettest days.** Write these dates down. In section 8 you will search local
  news for them to find out which streets actually flooded. That turns a statistic into
  evidence.

In [ ]:
from src import fetch_rainfall

rainfall = fetch_rainfall.fetch_daily_rainfall()
summary = fetch_rainfall.summarise_rainfall(rainfall)

In [ ]:
# Two charts: the yearly totals, and how lopsided the calendar is.
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 5))

yearly = rainfall.groupby(rainfall["date"].dt.year)["rainfall_mm"].sum()
left.bar(yearly.index, yearly.values, color="#2166ac")
left.axhline(yearly.mean(), color="#b2182b", linestyle="--",
             label=f"mean {yearly.mean():.0f} mm")
left.set_title("Total rainfall by year")
left.set_ylabel("mm")
left.legend()

monthly = rainfall.groupby(rainfall["date"].dt.month)["rainfall_mm"].mean() * 30
right.bar(monthly.index, monthly.values, color="#2166ac")
right.set_title("Average rainfall by month")
right.set_xlabel("month")
right.set_ylabel("mm")
right.set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

print("Look at the right-hand chart. Nearly everything lands in Jun-Sep.")
print("That concentration, not the annual total, is what overwhelms drainage.")

---
## 2. The shape of the ground

Water goes downhill. To know where it collects, we need to know the shape of the land.

We download **elevation tiles** — map tiles where each pixel's colour encodes the height
of the ground rather than a picture of it. The encoding is:

```
height in metres = (red x 256 + green + blue / 256) - 32768
```

The data comes originally from NASA's Shuttle Radar Topography Mission, which mapped
most of the planet's land surface from the Space Shuttle in 2000. Roughly 30 m per pixel.

This takes a couple of minutes. Each dot is one tile.

In [ ]:
from src import fetch_elevation

raw_dem, bounds = fetch_elevation.fetch_dem()

dem, _ = fetch_elevation.crop_to_study_area(raw_dem, bounds)
dem = fetch_elevation.fill_missing_elevations(dem)

# How much ground does one pixel cover? Needed for the slope and wetness maths
# to come out in real units.
lat_span_m = ((config.STUDY_AREA["max_lat"] - config.STUDY_AREA["min_lat"])
              * features.METRES_PER_DEGREE_LAT)
pixel_m = lat_span_m / dem.shape[0]

print(f"\nGrid: {dem.shape[0]} x {dem.shape[1]} pixels, about {pixel_m:.0f} m each")
print(f"Lowest point  {dem.min():.0f} m")
print(f"Highest point {dem.max():.0f} m")
print(f"Total relief  {dem.max() - dem.min():.0f} m")

In [ ]:
plt.figure(figsize=(10, 9))
plt.imshow(dem, cmap="terrain",
           extent=[config.STUDY_AREA["min_lon"], config.STUDY_AREA["max_lon"],
                   config.STUDY_AREA["min_lat"], config.STUDY_AREA["max_lat"]])
plt.colorbar(label="elevation (m)")
plt.title("Jaipur — elevation")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()

print("The dark ridges are the Aravalli hills. The pale ground between them is where")
print("the city sits, and where water from those hills has to go.")

---
## 3. Where does the water go?

This is the scientific core. Four steps, in a strict order.

**Step 1 — fill the sinks.** Elevation data has errors. One pixel wrongly 2 m too low
becomes a pit with no downhill neighbour; water flowing in vanishes and everything
downstream is wrong. So we "fill" every hollow first, like pouring water in until it can
spill over the lowest rim. The algorithm is called Priority-Flood.

**Step 2 — flow direction.** For each cell, which of its 8 neighbours does water go to?
Whichever is steepest downhill. This is the D8 method. Note that diagonal neighbours are
1.414 times further away, so we compare *slope*, not raw drop — forgetting this is the
classic bug and it skews every flow path diagonally.

**Step 3 — flow accumulation.** How many upstream cells drain through each cell? Process
from highest to lowest and pass the running total downstream. A cell with accumulation
5 catches its own rain; a cell with 5,000 is a drainage channel.

**Step 4 — Topographic Wetness Index.**

$$ TWI = \ln\left(\frac{a}{\tan\beta}\right) $$

where *a* is upslope area per unit contour width and *β* is the slope. The fraction gets
large when a lot of water arrives (big top) **and** the ground is flat so it leaves slowly
(small bottom). Somewhere that is both low and flat scores very high — which describes
almost every chronic waterlogging spot in any city.

This cell takes a minute or two. It is doing real work on a few hundred thousand pixels.

In [ ]:
terrain_result = terrain.analyse_terrain(dem, pixel_m)

print(f"Deepest depression found   {terrain_result['depression_depth'].max():.2f} m")
print(f"Peak flow accumulation     {terrain_result['flow_accumulation'].max():.0f} cells")
print(f"TWI range                  {terrain_result['twi'].min():.1f} to {terrain_result['twi'].max():.1f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
extent = [config.STUDY_AREA["min_lon"], config.STUDY_AREA["max_lon"],
          config.STUDY_AREA["min_lat"], config.STUDY_AREA["max_lat"]]

axes[0, 0].imshow(terrain_result["slope"], cmap="magma", extent=extent, vmax=15)
axes[0, 0].set_title("Slope — dark is flat, bright is steep")

# Log scale: flow accumulation spans several orders of magnitude, so on a linear
# scale you would see one bright line and nothing else.
axes[0, 1].imshow(np.log1p(terrain_result["flow_accumulation"]),
                  cmap="Blues", extent=extent)
axes[0, 1].set_title("Flow accumulation (log scale) — the natural drainage network")

axes[1, 0].imshow(terrain_result["twi"], cmap="YlGnBu", extent=extent)
axes[1, 0].set_title("Topographic Wetness Index — bright = water collects")

axes[1, 1].imshow(terrain_result["depression_depth"], cmap="Reds", extent=extent, vmax=2)
axes[1, 1].set_title("Depression depth — closed hollows with no outlet")

for ax in axes.ravel():
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")

plt.tight_layout()
plt.show()

**Stop and look at the top-right panel.** You did not tell the computer where Jaipur's
rivers and drains are. It worked out the drainage network purely from the shape of the
ground. Compare it to a satellite view of the city — the branching pattern should follow
real watercourses.

That is a genuine check that the hydrology is working, and it is a good thing to be able
to point at when explaining the project.

---
## 4. The city itself

Terrain tells you where water *would* go on bare ground. Now we need the city: what is
built there, and what drainage exists.

All of this comes from **OpenStreetMap**, mapped by volunteers, queried through Overpass.

**Be honest about the limitation.** OSM coverage in Jaipur is uneven. A cell with no
mapped drain might have no drain — or might just be somewhere nobody has surveyed. This
matters more than it sounds, because mapping effort tends to correlate with affluence,
which could bias the model *against* finding risk in exactly the poorly-served areas the
project is meant to identify. Say this out loud when you present the work.

The buildings query is the slow one. Give it a few minutes.

In [ ]:
from src import fetch_osm

osm_layers = fetch_osm.fetch_all_osm()

print()
for layer_name, items in osm_layers.items():
    print(f"  {layer_name:12s} {len(items):6d}")

---
## 5. Turning geography into a table

Right now you have four incompatible things: a pixel array of elevations, lists of lines
for drains and roads, and a scatter of building points. You cannot do statistics on that.

So we lay a **grid of 300 m squares** over the city and ask the same questions of each
square: how high, how flat, how wet, how many buildings, how much road, how far to a drain.
That grid becomes an ordinary table — one row per square, one column per measurement.

This step is called rasterisation and it is most of what applied GIS work actually is.

One detail worth knowing: a degree of longitude at Jaipur's latitude is about 99 km, not
111 km, because lines of longitude converge towards the poles. Ignoring that stretches
your map east-west by 11% and quietly corrupts every distance. The code handles it.

In [ ]:
grid = features.build_grid()
print(f"Grid: {grid['n_rows']} rows x {grid['n_cols']} cols "
      f"= {grid['n_rows'] * grid['n_cols']} cells of {grid['cell_size_m']} m")

table = features.build_feature_table(terrain_result, osm_layers, grid)

table.head()

In [ ]:
# Always look at your data before modelling it. Distributions reveal problems
# that summary statistics hide.
urban = table[table["is_urban"]]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
columns = [
    ("twi", "Topographic wetness index"),
    ("depression_depth_m", "Depression depth (m)"),
    ("relative_elevation_m", "Height relative to surroundings (m)"),
    ("building_density_per_ha", "Buildings per hectare"),
    ("drain_distance_m", "Distance to nearest drain (m)"),
    ("unpaved_fraction", "Fraction of roads unpaved"),
]

for ax, (column, label) in zip(axes.ravel(), columns):
    ax.hist(urban[column].dropna(), bins=50, color="#2166ac", edgecolor="white", linewidth=0.4)
    ax.set_title(label, fontsize=10)

plt.tight_layout()
plt.show()

print("Notice how skewed several of these are — a long tail with most cells bunched")
print("at the left. That skew is exactly why the next step ranks values instead of")
print("scaling them linearly.")

---
## 6. Scoring

Now we combine everything. The problem: you cannot add a slope in degrees to a building
count. Different units, different scales — whichever has bigger numbers would dominate
for no good reason.

So every variable is converted to a **0–1 percentile rank**: "what fraction of cells score
lower than this one?"

**Why rank and not min-max scaling?** Look at the flow accumulation histogram above. It is
brutally skewed. Min-max would squash 99% of the city into the bottom 1% of the scale and
your map would show one bright line against a flat background.

**The cost of ranking, which you must state:** it throws away magnitude. This index can
say a cell is worse than 90% of Jaipur. It can never say a cell is in absolute danger.
It ranks Jaipur against itself. That makes it a prioritisation tool, not a hazard
assessment, and the difference matters.

The three components are then blended using the weights in `src/config.py`.

In [ ]:
table = index.compute_component_scores(table)
table = index.compute_risk_index(table)

---
## 7. Does the answer survive your own assumptions?

**This is the most important cell in the notebook and almost nobody does it.**

You chose hazard = 0.50. A fair critic says: "if you had chosen 0.30 you would get a
different map, so why should I believe this one?" That is a completely reasonable
objection and the answer is not to argue — it is to test.

We re-run the whole index 200 times with every weight randomly jiggled by up to ±50%.
Cells that stay in the worst 10% under almost every weighting are **robust findings** you
can defend. Cells that jump around are artefacts of your assumptions, and reporting them
as results would be overclaiming.

In [ ]:
# This cell BOTH runs the sensitivity analysis and charts it.
# They are together on purpose: the chart reads a column that only
# exists after the analysis runs, so splitting them lets you skip
# one and get a confusing KeyError.

table = index.sensitivity_analysis(table, n_trials=200)

robust = table[table["is_urban"]]["rank_stability"].dropna()

plt.figure(figsize=(10, 5))
plt.hist(robust, bins=50, color="#2166ac", edgecolor="white", linewidth=0.4)
plt.axvline(0.95, color="#b2182b", linestyle="--", linewidth=2,
            label="0.95 — robust threshold")
plt.xlabel("Fraction of weightings in which this cell was in the worst 10%")
plt.ylabel("number of cells")
plt.title("How much does each cell's risk depend on the weights you chose?")
plt.legend()
plt.show()

n_robust = (robust >= 0.95).sum()
print(f"{n_robust} cells are high risk under 95%+ of plausible weightings.")
print("Those are the ones to report. The rest are weight-dependent — say so.")

In [ ]:
top_areas = index.top_risk_areas(table, n=20)

print("TOP 20 PRIORITY SITES")
print("=" * 78)
for rank, (_, site) in enumerate(top_areas.iterrows(), start=1):
    print(f"{rank:2d}. {site['lat']:.4f}, {site['lon']:.4f}   "
          f"risk {site['risk_index']:5.1f}   "
          f"robust {site['rank_stability'] * 100:3.0f}%   "
          f"{site['building_count']:4.0f} buildings   "
          f"{site['drain_distance_m']:5.0f} m to drain")

print()
print("Paste any of those coordinate pairs into Google Maps to see what is there.")
print("Do this. If a site is obviously a lake or a quarry, that is a finding about")
print("your model, and you should investigate rather than ignore it.")

---
## 8. The map

One self-contained HTML file. Click any cell to see the numbers behind its colour.

This is the artefact you can actually show someone — a municipal engineer, a teacher, an
admissions reader. It is also the version of your work that invites people to check it,
which is the point.

In [ ]:
risk_map = mapping.build_risk_map(
    table, grid,
    drains=osm_layers["drains"],
    top_areas=top_areas,
    synthetic=False,
)

risk_map

In [ ]:
# Save everything, then download it out of Colab.
table.to_csv("outputs/jaipur_flood_risk_cells.csv", index=False)
top_areas.to_csv("outputs/top_priority_sites.csv", index=False)

mapping.build_diagnostic_maps(table, grid)

from google.colab import files
files.download("outputs/jaipur_flood_risk_map.html")

---
## 9. The part that makes this real

Everything so far produces a plausible map. **A plausible map is not a result.**

The question anyone serious will ask is: *how do you know it is right?*

To answer it you need ground truth — places in Jaipur that really waterlogged, and places
that really did not, collected by you, with sources. Read
`docs/HOW_TO_COLLECT_GROUND_TRUTH.md` and build `data/waterlogging_observations.csv`.

**The trap:** you need the places that did NOT flood. With only flooded points, a model
that calls the whole city high-risk scores perfectly. Negative examples are what make the
test mean anything, and they are harder to find because nobody photographs a dry road.

Aim for 20 of each, minimum.

Go back to section 1 for the ten wettest days on record and search local news — including
Hindi coverage, which is far more detailed on local waterlogging — for those exact dates.

In [ ]:
# Run this once you have filled in data/waterlogging_observations.csv

if config.GROUND_TRUTH_CSV.exists():
    report = validate.full_validation_report(config.GROUND_TRUTH_CSV, table, grid)
else:
    print("No observations file yet.")
    print("The index is UNVALIDATED until you make one.")
    print("See docs/HOW_TO_COLLECT_GROUND_TRUTH.md")

### Reading your AUC honestly

| AUC | What it means |
|---|---|
| 0.50 | No better than a coin flip. Your index has no signal. |
| 0.65 | Weak but real. |
| 0.75 | Respectable for a screening tool built from open data. |
| 0.85 | Strong. |
| 1.00 | Suspect a bug or data leakage before celebrating. |

**A validated 0.71 is worth far more than an unvalidated claim of 95%.** If the number is
low, report it and investigate why — is 300 m too coarse? Is your ground truth shaky? Is
there drainage OSM does not know about? Those are interesting questions, and being the
person who asks them is the whole point.

Also look at the **baseline comparison**. If "relative elevation" alone matches your full
composite index, the extra machinery is not earning its keep. Say so. That is a real
finding, not a failure.

---
## 10. Optional: the supervised model

Once you have 40+ observations you can train an actual classifier and ask whether it
beats the physics-based index.

**Do not skip the honesty check here.** With 40 points and 10 features, a random forest
will happily memorise your data and report near-perfect accuracy that means nothing.
Cross-validation is what stops you fooling yourself: train on part of the data, test on
the part the model has never seen, repeat.

In [ ]:
# Requires a filled-in observations file with at least ~40 rows.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

if config.GROUND_TRUTH_CSV.exists():
    observations = validate.load_observations(config.GROUND_TRUTH_CSV)
    scored = validate.attach_scores(observations, table, grid)

    lookup = table.set_index(["row", "col"])
    feature_columns = ["twi", "depression_depth_m", "relative_elevation_m",
                       "slope_deg", "flow_accumulation", "drain_distance_m",
                       "building_density_per_ha", "unpaved_fraction"]

    rows = []
    for _, observation in scored.iterrows():
        r, c = features.latlon_to_cell(observation["lat"], observation["lon"], grid)
        if r is not None and (r, c) in lookup.index:
            rows.append(lookup.loc[(r, c)][feature_columns])

    X = pd.DataFrame(rows).values
    y = scored["flooded"].values[:len(rows)]

    if len(X) >= 30 and len(set(y)) == 2:
        model = RandomForestClassifier(
            n_estimators=300,
            # Shallow trees and a minimum leaf size. With this little data, an
            # unconstrained forest memorises rather than learns.
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
        )

        splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=splitter, scoring="roc_auc")

        print(f"Cross-validated AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")
        print(f"Index AUC for comparison: {validate.roc_auc(scored, verbose=False):.3f}")
        print()
        print("If the forest does NOT beat the index, that is a legitimate and")
        print("interesting result. With 40 data points the physics-based index")
        print("encodes more knowledge than the model can learn from scratch.")

        model.fit(X, y)
        importance = pd.Series(model.feature_importances_,
                               index=feature_columns).sort_values(ascending=False)
        print("\nWhat the model leaned on:")
        print(importance.to_string())
    else:
        print(f"Only {len(X)} usable observations. Collect more before training.")
else:
    print("Collect ground truth first — see section 9.")

---
## What to do next

**Make it real.** The single highest-value thing you can do is section 9. Forty honest
observations turn this from a nice map into a validated result.

**Take it to someone.** The Jaipur Municipal Corporation (Greater and Heritage) has
engineers who deal with this every monsoon. So does the Rajasthan State Disaster
Management Authority. Send the map. Ask whether the priority sites match what they see
on the ground. Either answer is valuable: agreement validates the model, disagreement
teaches you what the model is missing. This is also the step almost no student takes.

**Then improve it.** In rough order of payoff:

1. Census 2011 ward amenity data instead of infrastructure proxies
2. Sentinel-1 radar imagery from just after a big storm — radar sees through cloud and
   standing water shows as dark. That gives real observed flood extent and turns this
   into genuine supervised learning
3. Finer grid in the areas that matter most
4. Drainage capacity, if the municipal corporation will share it

**Write down what you find.** Including the parts that did not work. A short honest
write-up of a modest validated result is worth more than a grand claim nobody can check.